I'm going to start very simple, i.e. using logistic regression on age whether the patient will develop sepsis


In [ ]:
import pandas as pd
import numpy as np
from loguru import logger
from data_preprocessing.utils.data_handling import load_parquet_table
from pathlib import Path

import matplotlib.pyplot as plt

In [ ]:
notebook_path = Path.cwd()
project_root = notebook_path.parents[1]

In [ ]:
dataset_cat_path: Path = project_root / Path(
    "datasets/raw_and_shitty/merged_tables_cats.parquet")

df_dataset_cats = load_parquet_table(parquet_path=dataset_cat_path, verbose=False)
df_dataset_cats.head()

In [ ]:
dataset_ts_path: Path = project_root / Path(
    "datasets/raw_and_shitty/dataset_ts.parquet")

df_dataset_ts = load_parquet_table(parquet_path=dataset_ts_path, verbose=False)
df_dataset_ts.head()

In [ ]:
#ids_sepsis_patients = df_dataset_ts.loc[df_dataset_ts["sepsis_patient"] == True, ["patientunitstayid"]]["patientunitstayid"].unique().tolist()

In [ ]:
ids_patients_cats = df_dataset_cats["patientunitstayid"].unique().tolist()
ids_patients_ts = df_dataset_ts["patientunitstayid"].unique().tolist()

ids_patients_cats == ids_patients_ts

In [ ]:
df_sepsis_features = df_dataset_cats.loc[:, ["patientunitstayid", "age", "gender", "vent", "apachedxgroup"]]

# this is just a hack for now, will be cleaned up in the used dataset
df_sepsis_features["age" ] = pd.to_numeric(df_sepsis_features["age"], errors="coerce")

#df_sepsis_features = df_sepsis_features[df_sepsis_features["age"] != "> 89" ]


In [ ]:
df_sepsis_target = df_dataset_ts.loc[:, ["patientunitstayid", "sepsis_patient"]]
df_sepsis_target = df_sepsis_target.drop_duplicates(subset="patientunitstayid")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder

In [ ]:
X = df_sepsis_features.set_index("patientunitstayid")
y = df_sepsis_target.set_index("patientunitstayid")

In [ ]:
enc = OneHotEncoder(handle_unknown='ignore')
enc.fit(X)

enc.categories_

In [ ]:
y["sepsis_patient"].to_numpy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(enc.transform(X), y["sepsis_patient"].to_numpy(), test_size=0.2, random_state=42)


In [ ]:
model = LogisticRegression(penalty="l2",class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

In [ ]:
model.get_params()

In [ ]:
predictions = model.predict(X_test)
print(classification_report(y_test, predictions))